In [ ]:
import sys, os
from pathlib import Path
raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'modelos').exists())
sys.path.insert(0, str(raiz)); sys.path.insert(0, str(raiz / 'src' / 'dados'))
os.chdir(raiz / 'notebooks' / 'modelos')   # mantém os '../../' do resto funcionando
import numpy as np, pandas as pd, joblib
from scipy.optimize import brentq
from src.modelos.dados import ler_base, preparar
from src.modelos.variaveis import criar_variaveis
from src.modelos.treino import _para_modelo
from src.modelos.prever import juro_justo, LGD_POR_PAIS, LGD_PADRAO

pac = joblib.load('../../modelos_salvos/pacote_prever_v1.joblib')
base = criar_variaveis(preparar(ler_base()), referencia=pac['referencia'])
base = base[~base['fora_da_auditoria']]          # sem reestruturados

base['pd'] = pac['modelo'].predict_proba(_para_modelo(base))[:, 1]
mes = pd.to_datetime(base['data_emprestimo']).dt.to_period('M').dt.to_timestamp()
base['euribor'] = (mes.map(pac['euribor_12m']).fillna(0) / 100).clip(lower=0)
base['lgd'] = base['pais'].map(LGD_POR_PAIS).fillna(LGD_PADRAO)
base['cobrado'] = base['juro_aa'] / 100

def media_justo(d, m):
    return np.mean([juro_justo(int(r.prazo_meses), r.pd, r.lgd, r.euribor + m)
                    for r in d.itertuples()])

margens = {}
linhas = []
linhas = []
for ano, d in base.groupby('ano'):
    d = d.sample(min(len(d), 1000), random_state=0)
    alvo = d['cobrado'].mean()
    f = lambda m: media_justo(d, m) - alvo
    grade = {f'justo_m{int(m*100)}%': round(f(m) + alvo, 3) for m in (-0.05, 0.0, 0.05, 0.10)}
    m = brentq(f, 0.0, 1.0) if f(0.0) < 0 < f(1.0) else None
    linhas.append({'ano': ano, 'cobrado': round(alvo, 3), 'pd': round(d['pd'].mean(), 3),
                   **grade, 'margem': None if m is None else round(m, 4)})
print(pd.DataFrame(linhas).to_string(index=False))




 ano  cobrado    pd  justo_m-5%  justo_m0%  justo_m5%  justo_m10%  margem
2009    0.293 0.383       0.137      0.188      0.249       0.309  0.0873
2010    0.325 0.352       0.186      0.158      0.217       0.275  0.1427
2011    0.255 0.234       0.370      0.085      0.139       0.193     NaN
2012    0.238 0.210       2.332      0.060      0.113       0.166     NaN
2013    0.263 0.343       2.271      0.075      0.129       0.183     NaN
2014    0.293 0.524       1.402      0.231      0.294       0.356     NaN
2015    0.345 0.631       1.094      0.234      0.296       0.358     NaN
2016    0.468 0.544       1.273      0.216      0.278       0.339     NaN
